In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import shutil
from pathlib import Path

In [3]:
PROJECT_ROOT = Path.cwd().parents[0]
os.chdir(PROJECT_ROOT)

In [4]:
from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.vectorstores import VectorStoreRetriever

/var/folders/1w/k2xkpg3s50n1nnp_jrdp_pdw0000gn/T/ipykernel_30376/115363839.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [5]:
from src.embeddings import get_embedding_model
from src.loaders import load_documents, split_documents

In [6]:
load_dotenv()

DEFAULT_VECTORSTORE_DIR = Path("data/vectorstore")
DEFAULT_DOCUMENTS_DIR = Path("data/documents")
INDEX_NAME = "index"

In [7]:
documents_dir: str | Path = str(DEFAULT_DOCUMENTS_DIR)
persist_directory: str | Path = str(DEFAULT_VECTORSTORE_DIR)
chunk_size: int | None = int(os.getenv("CHUNK_SIZE", "1000"))
chunk_overlap: int | None = int(os.getenv("CHUNK_OVERLAP", "150"))

documents = load_documents(documents_dir)

In [8]:
len({doc.metadata.get("filename") for doc in documents})

3

In [9]:
num_files = len({doc.metadata.get("filename") for doc in documents})

In [10]:
chunks = split_documents(
    documents,
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
)

In [11]:
embeddings = get_embedding_model()
vectorstore = FAISS.from_documents(chunks, embeddings)

In [12]:
vectorstore

In [13]:
vectorstore.index.ntotal

6

In [14]:
len(chunks)

6

In [15]:
persist_path = Path(persist_directory)
persist_path.mkdir(parents=True, exist_ok=True)
vectorstore.save_local(str(persist_path), index_name=INDEX_NAME)

In [16]:
# How to load a saved vector store

embeddings = get_embedding_model()
vectorstore = FAISS.load_local(
    str(persist_directory),
    embeddings,
    index_name=INDEX_NAME,
    allow_dangerous_deserialization=True,
)

In [17]:
# How to use it as retriever
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": int(os.getenv("TOP_K", "4"))
    }
)

In [18]:
question = "what is the name of the company?"

retrieved_docs = retriever.invoke(question)

In [19]:
# Format the retrieved chunks for giving it to prompt
parts = []
for i, doc in enumerate(retrieved_docs, start=1):
    filename = doc.metadata.get("filename") or doc.metadata.get("source", "unknown")
    page = doc.metadata.get("page")
    location = f"{filename}"
    if page is not None:
        location = f"{filename} (page {page})"
    parts.append(f"[Source {i}: {location}]\n{doc.page_content}")

In [20]:
final_parts = "\n\n".join(parts)

In [21]:
final_parts

'[Source 1: company_handbook.pdf (page 2)]\n3. Remote Work\nEmployees may work remotely up to 3 days per week after completing their first 90 days with the\ncompany. Remote work days must be agreed with the team lead in advance. Employees working\nremotely must remain reachable on Slack and email during core collaboration hours. Fully remote\nroles are available only for approved positions.\n4. Employee Benefits\nBenefits include health insurance starting on the first day of employment, a learning stipend of $500\nper year, and a home-office stipend of $300 for employees who work remotely at least 2 days each\nweek. The company also provides access to an employee assistance program for mental health\nsupport.\n5. Performance Reviews\nFormal performance reviews happen twice each year, in June and December. Employees set goals\nwith their managers at the start of each review cycle. Promotion discussions are typically held during\nthe December review window.\nPage 2\n\n[Source 2: company_